# Residential location: dwelling

Lower level of the residential location model: which dwelling a moving household chooses within its destination PUMA.

Article table: `est_dwelling`. This notebook goes from the raw data to the estimates of this model:

1. Raw data
2. Data preparation
3. Estimation data
4. Descriptive statistics
5. Correlation of the explanatory variables (heatmaps, VIF)
6. Model specification (the folder's `model/estimate_residential_dwelling.py`, unchanged)
7. Log-likelihood at the article coefficients
8. Biogeme estimation
9. Results: article format, fit, comparison with the article, correlation of the estimates
10. Hold-out sample

**Kernel:** select *Python (biogeme)* (`~/.venvs/biogeme`) in VS Code, then *Run All*. The switches in the next cell decide what is recomputed.

In [ ]:
RUN_DATA_PREP  = False   # True: rebuild the estimation tables from input/data/raw (section 2); a copy of the provided tables is kept first
RUN_CHECK      = True    # log-likelihood at the article coefficients
RUN_ESTIMATION = True    # Biogeme estimation, written to output/
RUN_HOLDOUT    = True    # log-likelihood of the hold-out sample at your estimates

import os, sys, json, glob, inspect, importlib, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import biogeme.biogeme as bio
from biogeme.parameters import Parameters
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "residential_dwelling"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", FOLDER)

## 1. Raw data

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**ACS PUMS 2014-2018 5-year, household and person files** — U.S. Census Bureau
- Where: <https://www2.census.gov/programs-surveys/acs/data/pums/2018/5-Year/>
- How: Download csv_h<st>.zip and csv_p<st>.zip for st = dc, de, md, pa, va, wv from the 2018 5-Year folder (CSV, not SAS). Variables used: SERIALNO, ST, PUMA, MIGPUMA, MIGSP, POWPUMA, POWSP, MIG, MV, RELP, AGEP, ESR, JWMNP, SCHG, NP, TYPE, BDSP, BLD, TEN, GRNTP, SMOCP, VALP, HINCP, VEH, YBL, WGTP; ADJINC (income adjustment) and ADJHSG (housing-cost adjustment) are columns of the same files.
- Put in: `00_shared_data/raw/acs_pums_2014_2018_5yr/`
- Files: `csv_hdc.zip`, `csv_hde.zip`, `csv_hmd.zip`, `csv_hpa.zip`, `csv_hva.zip`, `csv_hwv.zip`, `csv_pdc.zip`, `csv_pde.zip`, `csv_pmd.zip`, `csv_ppa.zip`, `csv_pva.zip`, `csv_pwv.zip`

**2010 Census tract to 2010 PUMA relationship file** — U.S. Census Bureau
- Where: <https://www2.census.gov/geo/docs/maps-data/data/rel/2010_Census_Tract_to_2010_PUMA.txt>
- How: Download the national text file.
- Put in: `00_shared_data/raw/geo/`
- Files: `2010_Census_Tract_to_2010_PUMA.txt`

**PUMA to MIGPUMA and POWPUMA crosswalk (2010 geography)** — IPUMS USA
- Where: <https://usa.ipums.org/usa/resources/volii/puma_migpuma1_pwpuma00.xls>
- How: Download the xls.
- Put in: `00_shared_data/raw/geo/`
- Files: `puma_migpuma1_pwpuma00.xls`

**Census Gazetteer, national tract file (land area, internal points)** — U.S. Census Bureau
- Where: <https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html>
- How: Gazetteer files, census tracts, national file (the project uses Gaz_tracts_national.txt).
- Put in: `00_shared_data/raw/geo/`
- Files: `Gaz_tracts_national.txt`

**MSTM zone system, zone polygons and peak car skim (Maryland Statewide Transportation Model)** — Maryland Department of Transportation State Highway Administration (MDOT SHA), provided to the project
- Where: <https://roads.maryland.gov>
- How: Not a public download: the 1,588-zone MSTM system (zoneSystem.csv), zone polygons (SMZ_RMZ_02152011inMSTM_EPSG26918.shp) and the AM-peak auto travel-time skim were provided by MDOT. Request them from MDOT SHA (MSTM / travel forecasting).
- Put in: `00_shared_data/raw/mstm/`
- Files: `zoneSystem.csv`, `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.shx/.dbf/.prj`, `auto_peak_time.csv.gz`

**CPI-U, U.S. city average, all items (annual averages 2014-2023)** — U.S. Bureau of Labor Statistics
- Where: <https://data.bls.gov/timeseries/CUUR0000SA0>
- How: Series CUUR0000SA0, annual averages; the values are typed in rc.py (CPI dict) and used to express incomes in 2016 dollars.
- Put in: `00_shared_data/raw/cpi/`

In [ ]:
# files of input/data/raw (the model's own extracts; shared sources are in ../00_shared_data)
raw = os.path.join(FOLDER, "input", "data", "raw")
display(nt.files_table(raw))
for f in sorted(glob.glob(os.path.join(raw, "*")))[:6]:
    head, n = nt.preview(f)
    if head is not None:
        print(f"\n{os.path.basename(f)}" + (f"  ({n:,} rows)" if n else ""))
        display(head)

In [ ]:
print(open(os.path.join(raw, "README.md")).read() if os.path.exists(os.path.join(raw, "README.md")) else "no README in input/data/raw")

## 2. Data preparation

Upstream stages (outputs provided in `00_shared_data` and `input/data/raw`; run them only for a full rebuild from the raw data, commands from the model folder):

- **Step 1.** SILO base-year microdata (upstream project stage; output provided (redo only for a rebuild from raw data; writes into the project folders: back them up first))  
  `cd ../../../2_silo/input_build` → `/opt/homebrew/anaconda3/bin/python3 build_microdata.py && /opt/homebrew/anaconda3/bin/python3 build_base_v2.py`

Package steps, run below when `RUN_DATA_PREP = True`: they rebuild `input/data/estimation` and `input/data/holdout` with the original random draws and stop if the log-likelihood of the original run is not reproduced; `check_against_provided.py` must end with *ALL TABLES MATCH*.

**Step 2.** Geography: zone / PUMA / MIGPUMA tables (package copy; writes 00_shared_data/resloc_work/geo_*.csv). Needs jj_2016.csv in 00_shared_data/synthetic_population_2016 (copy it from 2_silo/input/microData), the tract-PUMA file in 00_shared_data/county_attributes (rc.RAW) and the Gazetteer file at rc.GAZ

In [ ]:
if RUN_DATA_PREP:
    nt.run('/opt/homebrew/anaconda3/bin/python3 s01_geography.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 3.** ACS PUMS movers and the year's dwelling market, estimation file 2014-2018 (package copy; writes 00_shared_data/resloc_work/pums_*_est.parquet)

In [ ]:
if RUN_DATA_PREP:
    nt.run('/opt/homebrew/anaconda3/bin/python3 s03_pums.py est', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 4.** input_check, part 1: keep a copy of the provided estimation and hold-out tables (about 0.1 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('rm -rf provided_estimation provided_holdout && cp -R estimation provided_estimation && cp -R holdout provided_holdout', cwd=os.path.join(FOLDER, 'input/data'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 5.** Build the estimation and hold-out tables from the inputs (same calls and random draws as the original run; asserts that the LL at the original estimates is reproduced) (about 0.5 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('~/.venvs/biogeme/bin/python build_estimation_data.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 6.** input_check, part 2: compare the rebuilt tables with the provided ones (rows, column sums, largest difference); must print ALL TABLES MATCH (about 0.5 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('~/.venvs/biogeme/bin/python check_against_provided.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

## 3. Estimation data

The table exactly as used in the estimation, and its data dictionary.

In [ ]:
est = os.path.join(FOLDER, "input", "data", "estimation")
display(nt.files_table(est))
dd = pd.read_csv(os.path.join(est, "data_dictionary.csv"))
dd

In [ ]:
import estimate_residential_dwelling as M
df = pd.read_parquet(M.DATA)
print(f"{df.shape[0]:,} rows (decision makers), {df.shape[1]:,} columns")
df.head()

## 4. Descriptive statistics

In [ ]:
alts = nt.alt_columns(df)
J = len(next(iter(alts.values())))
print(f"{J} alternatives per decision maker (columns <variable>_<j>)")
if "av" in alts:
    n_av = df[alts["av"]].sum(axis=1)
    print("available alternatives per decision maker:", n_av.describe().round(1).to_dict())
if "choice" in df:
    c = df["choice"].value_counts().sort_index()
    ax = (100 * c / c.sum()).plot.bar(figsize=(min(14, 3 + 0.12 * J), 3), color="#2b6cb0", width=0.8)
    ax.set_xlabel("position of the chosen alternative"); ax.set_ylabel("%"); ax.set_title("Chosen alternative (position in the set)")
    ax.set_xticks(ax.get_xticks()[::max(1, J // 20)]); plt.show()
person = [c for c in df.columns if not any(c in v for v in alts.values())]
display(nt.explanatory(df[person]).describe().T.round(3))
if "weight" in df:
    ax = df["weight"].plot.hist(bins=60, figsize=(5, 3), color="#2b6cb0"); ax.set_title("Estimation weight"); plt.show()
    print("weight: mean %.3f, min %.3f, max %.3f" % (df.weight.mean(), df.weight.min(), df.weight.max()))

## 5. Model specification

The specification is the folder's script `model/estimate_residential_dwelling.py`, imported as it is. The start values are the article estimates.

In [ ]:
print(inspect.getsource(M.model))

In [ ]:
t0 = time.time()
database, ll = M.model()
used = nt.model_variables(ll)
print(f"database: {database.num_rows():,} rows; {len(used)} data columns used by the model; built in {time.time()-t0:.1f} s")

## 6. Correlation of the explanatory variables

Correlations of the variables that enter the utility, computed over the alternatives available to each decision maker
(alternative-specific variables) and over decision makers (person and household variables). |r| ≥ 0.7 and VIF ≥ 10 are
listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
stems = sorted({c.rsplit("_", 1)[0] for c in used if c.rsplit("_", 1)[0] in alts and c.rsplit("_", 1)[-1].isdigit()}
               - {"av", "chosen_set", "zone", "stype"})
long = nt.explanatory(nt.stack_alternatives(df, stems))
print(f"{len(stems)} alternative-specific variables, {len(long):,} available alternatives")
Ca = nt.corr_heatmap(long, "Alternative-specific variables (available alternatives)")
display(nt.high_pairs(Ca)); display(nt.vif(long))
plain = [c for c in used if c in df.columns and c not in sum(alts.values(), []) and c not in ("choice", "weight")]
if len(plain) > 1:
    Xp = nt.explanatory(df[plain]); Cp = nt.corr_heatmap(Xp, "Decision-maker variables"); display(nt.high_pairs(Cp))

## 7. Log-likelihood at the article coefficients

No estimation: the log-likelihood at the start values (the article estimates). Target: article -47688.2,
Biogeme -47688.1973 (N = 15163, K = 38).

In [ ]:
if RUN_CHECK:
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(__import__("tempfile").mkdtemp())
    t0 = time.time()
    b = bio.BIOGEME(database, ll, parameters=p); b.model_name = NAME + "_check"
    ll0 = b.calculate_init_likelihood()
    os.chdir(here)
    print(f"N {database.num_rows():,}   LL at the article coefficients: {ll0:,.4f}   (target -47688.1973)   {time.time()-t0:.0f} s")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

The same call as `model/estimate_residential_dwelling.py`: Biogeme with its defaults (read from `output/biogeme.toml`), start values = article estimates.
Report written to `output/residential_dwelling.html`, `.yaml`, and `output/residential_dwelling_estimates.csv`.

In [ ]:
if RUN_ESTIMATION:
    here = os.getcwd(); os.chdir(OUT)
    t0 = time.time()
    b = bio.BIOGEME(database, ll); b.model_name = NAME
    results = b.estimate()
    os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s"); print(results.short_summary())
    nt.estimates_table(results)[["parameter", "coefficient", "robust_se", "t", "p"]].rename(
        columns={"coefficient": "Value", "robust_se": "Robust std err.", "t": "Robust t-stat.", "p": "Robust p-value", "parameter": "Name"}
    ).to_csv(os.path.join(OUT, f"{NAME}_estimates.csv"), index=False)
else:
    y = sorted(glob.glob(os.path.join(OUT, f"{NAME}*.yaml")), key=os.path.getmtime)
    results = EstimationResults.from_yaml_file(y[-1]) if y else None
    print(f"RUN_ESTIMATION = False: read {os.path.basename(y[-1])}" if y else "RUN_ESTIMATION = False and no result in output/: sections 9-10 are skipped")

## 9. Results

### Estimates in the article format
Coefficient (stars: *** p < 0.001, ** p < 0.01, * p < 0.05), robust s.e., p.

In [ ]:
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### Goodness of fit

In [ ]:
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "N": database.num_rows() if "database" in dir() else None,
                                                   "K": results.number_of_free_parameters})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

In [ ]:
if RUN_HOLDOUT and results is not None:
    import tempfile
    est_data = M.DATA
    M.DATA = os.path.join(FOLDER, "input", "data", "holdout", 'dwelling_holdout.parquet')
    try:
        hdb, hll = M.model(start=results.get_beta_values())
    finally:
        M.DATA = est_data
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(tempfile.mkdtemp())
    b = bio.BIOGEME(hdb, hll, parameters=p); b.model_name = NAME + "_holdout"
    llh = b.calculate_init_likelihood(); os.chdir(here)
    print(f"hold-out: N {hdb.num_rows():,}   LL at your estimates {llh:,.2f}   mean LL per observation {llh/hdb.num_rows():.4f}")
    print(f"estimation sample: mean LL per observation {results.final_loglikelihood/database.num_rows():.4f}")
else:
    print("skipped (RUN_HOLDOUT = False or no estimates)")